In [ ]:
import marimo as mo

In [ ]:
import polars as pl
import numpy as np
from catboost import CatBoostClassifier
import pandas as pd

from metric import precision_at_recall, pr_curve

pl.Config.set_tbl_rows(-1)

# Изначальная идея была такова:
Мета модель на классике + Laya(https://huggingface.co/convaiinnovations/laya), но временные рамки и вычислительные мощности ноутбука не позволяют локально гонять Laya, а kaggle и collab я не люблю особо

In [ ]:
features_train = pl.read_parquet("data/features_train.parquet")
features_test = pl.read_parquet("data/features_test.parquet")

train = pl.read_csv("data/train.csv")
test = pl.read_csv("data/test.csv")

features_train = features_train.join(
    train.select("cookie_id", "window_start_ts"), on="cookie_id", how="left"
)
# бьем дни на 7 фолдов
days = features_train["window_start_ts"].unique().sort()
day_groups = np.array_split(days.to_numpy(), 7)

fold_map = {}
for fold_id, group in enumerate(day_groups):
    for d in group:
        fold_map[d] = fold_id

features_train = features_train.with_columns(
    pl.col("window_start_ts").replace(fold_map, return_dtype=pl.Int64).alias("fold")
)

features_train.group_by("fold").agg(pl.len().alias("n_cookies"), pl.col("target").mean().alias("bot_share")).sort("fold")

In [ ]:
feature_cols = [
    c for c in features_train.columns
    if c not in ("cookie_id", "target", "window_start_ts", "fold")
]
len(feature_cols), feature_cols

In [ ]:
X = features_train.select(feature_cols).to_pandas()
y = features_train["target"].to_numpy()
fold = features_train["fold"].to_numpy()

oof = np.zeros(len(y))

for f in range(7):
    is_valid = fold == f

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        random_seed=42,
        verbose=0,
    )
    model.fit(X[~is_valid], y[~is_valid])
    oof[is_valid] = model.predict_proba(X[is_valid])[:, 1]

oof[:10]

In [ ]:
score = precision_at_recall(y, oof)
print("P@R>=0.7:", round(score, 4))

# для сравнения уровень константы (если просто всем ставить одну оценку)
print("доля ботов в train (уровень константы):", round(y.mean(), 4))

In [ ]:
for q in range(7):
    is_val = fold == q
    fold_score = precision_at_recall(y[is_val], oof[is_val])
    print(f"фолд {q}: P@R>=0.7 = {round(fold_score, 4)}, кук = {is_val.sum()}, ботов = {y[is_val].sum()}")

In [ ]:
final_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    random_seed=42,
    verbose=0,
)
final_model.fit(X, y)

In [ ]:
X_test = features_test.select(feature_cols).to_pandas()
test_scores = final_model.predict_proba(X_test)[:, 1]

submission = pl.DataFrame({
    "cookie_id": features_test["cookie_id"],
    "score": test_scores,
})

In [ ]:
submission.write_csv("submission.csv")

Прогнал на степике, получил позорный результат. Попробую получить еще сильные фичи